CLEANING THE DATSET GIVEN FOR THE ANALYSIS


In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import FunctionTransformer


CLEANING


In [27]:

# Correct folder path (because notebook is already inside Token-Trend)
folder = Path("DATA")
files = sorted([f for f in folder.glob("*.csv") if not f.name.startswith("cleaned_")])

required_cols = ["date", "open", "high", "low", "close", "volume"]
numeric_cols = ["open", "high", "low", "close", "volume"]

summary = []

for file_path in files:
    raw = pd.read_csv(file_path)
    original_rows = len(raw)

    # Standardize column names
    raw.columns = raw.columns.str.strip().str.lower()

    # Check required columns
    missing_cols = [c for c in required_cols if c not in raw.columns]
    if missing_cols:
        summary.append({
            "file": file_path.name,
            "status": f"skipped (missing columns: {missing_cols})",
            "original_rows": original_rows,
            "cleaned_rows": 0,
            "rows_removed": original_rows
        })
        continue

    # Clean data directly on raw
    raw = raw.dropna(subset=required_cols)
    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
    for c in numeric_cols:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
    raw = raw.dropna(subset=required_cols)

    # Business rules
    raw = raw[
        (raw["open"] >= 0) &
        (raw["high"] >= 0) &
        (raw["low"] >= 0) &
        (raw["close"] >= 0) &
        (raw["volume"] >= 0) &
        (raw["high"] >= raw["low"])
    ].drop_duplicates()

    cleaned_rows = len(raw)
    removed_rows = original_rows - cleaned_rows

    # Save with changed file name: cleaned_<original>.csv
    cleaned_name = f"cleaned_{file_path.name}"
    cleaned_path = file_path.with_name(cleaned_name)
    raw.to_csv(cleaned_path, index=False)

    summary.append({
        "file": file_path.name,
        "new_file": cleaned_name,
        "status": "cleaned and saved as new filename",
        "original_rows": original_rows,
        "cleaned_rows": cleaned_rows,
        "rows_removed": removed_rows
    })

summary_df = pd.DataFrame(summary)
summary_df

,file,status,original_rows,cleaned_rows,rows_removed
0,file_importance_scores.csv,"skipped (missing columns: ['date', 'open', 'hi...",23,0,23


LOGISTIC REGRESSION 

FILE cleaned_coin_Bitcoin.csv
only logistic regresssion with normal feateures]

In [28]:
# Logistic Regression for ONE file only with linear terms
# Target: 1 if next day's return > 5% threshold, else 0
# Model form per feature: a1*x (no sqrt, no x^2, no interactions)
# Train/Test: first 3/4 rows for training, last 1/4 rows for testing

# -----------------------------
# 1) Select one file
# -----------------------------
file_path = "DATA/cleaned_coin_Bitcoin.csv"
df = pd.read_csv(file_path)

# -----------------------------
# 2) Helper for flexible column matching
# -----------------------------
def _norm(s: str):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def find_col(columns, candidates):
    norm_map = {_norm(c): c for c in columns}
    for cand in candidates:
        if cand in norm_map:
            return norm_map[cand]
    return None

open_col  = find_col(df.columns, ["open", "priceopen"])
high_col  = find_col(df.columns, ["high", "pricehigh"])
low_col   = find_col(df.columns, ["low", "pricelow"])
close_col = find_col(df.columns, ["close", "priceclose"])
vol_col   = find_col(df.columns, ["volume", "volumefrom", "volumeto", "totalvolume", "tradevolume"])

if any(c is None for c in [open_col, high_col, low_col, close_col, vol_col]):
    raise ValueError("Required columns not found (need open/high/low/close/volume).")

# -----------------------------
# 3) Build base features and target
# -----------------------------
base_features = ["open", "high", "low", "close", "volume"]
data = df[[open_col, high_col, low_col, close_col, vol_col]].copy()
data.columns = base_features
data = data.apply(pd.to_numeric, errors="coerce")

# Target: next-day movement with 5% threshold
threshold = 0.05
data["next_day_return"] = data["close"].shift(-1) / data["close"] - 1
data["target_up_next_day"] = (data["next_day_return"] > threshold).astype(int)
data = data.dropna().reset_index(drop=True)

X = data[base_features]
y = data["target_up_next_day"].astype(int)

# -----------------------------
# 4) 3/4 split (time order)
# -----------------------------
n = len(X)
split_idx = (3 * n) // 4
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# -----------------------------
# 5) Train model
# -----------------------------
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, C=10.0, random_state=42))
])
model.fit(X_train, y_train)

# -----------------------------
# 6) Accuracy
# -----------------------------
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

accuracy_df = pd.DataFrame([{
    "file": file_path,
    "rows_used": n,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "accuracy": accuracy
}])

# -----------------------------
# 7) Extract linear weights
# -----------------------------
clf = model.named_steps["clf"]
coef = clf.coef_.ravel()
intercept = clf.intercept_[0]
coef_map = dict(zip(base_features, coef))

linear_weight_df = pd.DataFrame({
    "file": file_path,
    "feature": base_features,
    "a1_linear": [coef_map[f] for f in base_features]
})
linear_weight_df["equation"] = linear_weight_df.apply(
    lambda r: f"{r['feature']}: {r['a1_linear']:.6f}*x",
    axis=1
)

degree_wise_weights_df = pd.DataFrame({
    "file": file_path,
    "feature": base_features,
    "degree": [1.0] * len(base_features),
    "term": base_features,
    "weight": [coef_map[f] for f in base_features]
})

# -----------------------------
# 8) Feature importance (linear only)
# -----------------------------
abs_coef = np.abs(coef)
importance_pct = np.zeros_like(abs_coef) if abs_coef.sum() == 0 else (abs_coef / abs_coef.sum()) * 100

feature_importance_df = pd.DataFrame({
    "file": file_path,
    "feature": base_features,
    "abs_weight": abs_coef,
    "importance_score_percent": importance_pct
}).sort_values("importance_score_percent", ascending=False).reset_index(drop=True)
feature_importance_df["rank"] = np.arange(1, len(feature_importance_df) + 1)
feature_importance_df = feature_importance_df[[
    "file", "rank", "feature", "abs_weight", "importance_score_percent"
]]

# -----------------------------
# 9) Display outputs
# -----------------------------
print("Accuracy:")
print(accuracy_df.to_string(index=False))

print("\nPer-feature linear weights (a1*x):")
print(linear_weight_df.to_string(index=False))

print("\nDegree-wise weights table:")
print(degree_wise_weights_df.to_string(index=False))

print("\nFeature Importance Scores:")
print(feature_importance_df.to_string(index=False))

Accuracy:
                         file  rows_used  train_rows  test_rows  accuracy
DATA/cleaned_coin_Bitcoin.csv       2990        2242        748  0.890374

Per-feature linear weights (a1*x):
                         file feature  a1_linear            equation
DATA/cleaned_coin_Bitcoin.csv    open   0.148600    open: 0.148600*x
DATA/cleaned_coin_Bitcoin.csv    high   2.149104    high: 2.149104*x
DATA/cleaned_coin_Bitcoin.csv     low  -2.394030    low: -2.394030*x
DATA/cleaned_coin_Bitcoin.csv   close   0.359647   close: 0.359647*x
DATA/cleaned_coin_Bitcoin.csv  volume  -0.164199 volume: -0.164199*x

Degree-wise weights table:
                         file feature  degree   term    weight
DATA/cleaned_coin_Bitcoin.csv    open     1.0   open  0.148600
DATA/cleaned_coin_Bitcoin.csv    high     1.0   high  2.149104
DATA/cleaned_coin_Bitcoin.csv     low     1.0    low -2.394030
DATA/cleaned_coin_Bitcoin.csv   close     1.0  close  0.359647
DATA/cleaned_coin_Bitcoin.csv  volume     1.0 vol

In [29]:
# Returns + average returns + standard deviation of returns (named volatility) for ONE file
folder = Path("DATA")
files = sorted(folder.glob("cleaned_*.csv"))

for file_path in files:
    df = pd.read_csv(file_path)

    # Optional: parse date if present
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date").reset_index(drop=True)

    # Make sure close is numeric
    df["close"] = pd.to_numeric(df["close"], errors="coerce")

    # Daily returns from close price
    df["returns"] = df["close"].pct_change()

    # Drop NaN returns (first row and any bad rows)
    returns = df["returns"].dropna()

    # Required metrics
    avg_returns = returns.mean()
    volatility = returns.std()  # std deviation of returns = volatility

    print(f"File: {file_path}")
    print(f"Average Returns: {avg_returns:.6f}")
    print(f"Volatility (Std Dev of Returns): {volatility:.6f}")

    # Optional summary table
    summary_df = pd.DataFrame([{
        "file": file_path,
        "rows_used": len(df),
        "returns_count": len(returns),
        "avg_returns": avg_returns,
        "volatility": volatility
    }])

    display(summary_df)

File: DATA/cleaned_coin_Aave.csv
Average Returns: 0.010243
Volatility (Std Dev of Returns): 0.086548


,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_Aave.csv,275,274,0.010243,0.086548


File: DATA/cleaned_coin_BinanceCoin.csv
Average Returns: 0.008493
Volatility (Std Dev of Returns): 0.080050


,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_BinanceCoin.csv,1442,1441,0.008493,0.08005


File: DATA/cleaned_coin_Bitcoin.csv
Average Returns: 0.002741
Volatility (Std Dev of Returns): 0.042639


...

,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_Uniswap.csv,292,291,0.008031,0.091299


File: DATA/cleaned_coin_WrappedBitcoin.csv
Average Returns: 0.003512
Volatility (Std Dev of Returns): 0.042857


,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_WrappedBitcoin.csv,888,887,0.003512,0.042857


File: DATA/cleaned_coin_XRP.csv
Average Returns: 0.004478
Volatility (Std Dev of Returns): 0.081566


,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_XRP.csv,2893,2892,0.004478,0.081566


file all


In [30]:
# Logistic Regression across ALL cleaned files (linear terms only)
# Target: 1 if next day's return > 5% threshold, else 0
# Train/Test: first 3/4 rows for training, last 1/4 rows for testing

folder = Path("DATA")
files = sorted(folder.glob("cleaned_*.csv"))

#to normalize the function like conveting to lower case 
def _norm(s: str):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def find_col(columns, candidates):
    norm_map = {_norm(c): c for c in columns}
    for cand in candidates:
        if cand in norm_map:
            return norm_map[cand]
    return None

base_features = ["open", "high", "low", "close", "volume"]

metrics_rows = []
weights_rows = []
feature_imp_rows = []
powers_rows = []
skipped_rows = []

for file_path in files:
    df = pd.read_csv(file_path)

    open_col  = find_col(df.columns, ["open", "priceopen"])
    high_col  = find_col(df.columns, ["high", "pricehigh"])
    low_col   = find_col(df.columns, ["low", "pricelow"])
    close_col = find_col(df.columns, ["close", "priceclose"])
    vol_col   = find_col(df.columns, ["volume", "volumefrom", "volumeto", "totalvolume", "tradevolume"])

    if any(c is None for c in [open_col, high_col, low_col, close_col, vol_col]):
        skipped_rows.append({
            "file": file_path.name,
            "reason": "missing one or more required columns"
        })
        continue

    data = df[[open_col, high_col, low_col, close_col, vol_col]].copy()
    data.columns = base_features
    data = data.apply(pd.to_numeric, errors="coerce")

    # Target: next-day return with 5% threshold
    threshold = 0.0433
    data["next_day_return"] = data["close"].shift(-1) / data["close"] - 1
    data["target_up_next_day"] = (data["next_day_return"] > threshold).astype(int)
    data = data.dropna().reset_index(drop=True)

    X = data[base_features]
    y = data["target_up_next_day"].astype(int)

    n = len(X)
    if n < 20:
        skipped_rows.append({
            "file": file_path.name,
            "reason": f"not enough usable rows ({n})"
        })
        continue

    split_idx = (3 * n) // 4
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, C=10.0, random_state=42))
    ])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    clf = model.named_steps["clf"]
    coef = clf.coef_.ravel()
    intercept = clf.intercept_[0]

    abs_coef = np.abs(coef)
    importance_pct = np.zeros_like(abs_coef) if abs_coef.sum() == 0 else (abs_coef / abs_coef.sum()) * 100

    feature_importance_df = pd.DataFrame({
        "feature": base_features,
        "weight": coef,
        "abs_weight": abs_coef,
        "importance_score_percent": importance_pct
    }).sort_values("importance_score_percent", ascending=False).reset_index(drop=True)
    feature_importance_df["rank"] = np.arange(1, len(feature_importance_df) + 1)

    metrics_rows.append({
        "file": file_path.name,
        "rows_used": n,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "accuracy": accuracy
    })

    w_row = {"file": file_path.name, "intercept": intercept}
    w_row.update({f"w_{f}": c for f, c in zip(base_features, coef)})
    weights_rows.append(w_row)

    powers_rows.append({
        "file": file_path.name,
        "feature": "intercept",
        "power_used": 0,
        "weight": intercept
    })
    for f_name, c in zip(base_features, coef):
        powers_rows.append({
            "file": file_path.name,
            "feature": f_name,
            "power_used": 1,
            "weight": c
        })

    for _, row in feature_importance_df.iterrows():
        feature_imp_rows.append({
            "file": file_path.name,
            "rank": int(row["rank"]),
            "feature": row["feature"],
            "weight": row["weight"],
            "abs_weight": row["abs_weight"],
            "importance_score_percent": row["importance_score_percent"]
        })

metrics_df = pd.DataFrame(metrics_rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
weights_df_all = pd.DataFrame(weights_rows)
powers_by_model_df = pd.DataFrame(powers_rows)
feature_importance_all_df = pd.DataFrame(feature_imp_rows)
skipped_df = pd.DataFrame(skipped_rows)

print("Model performance by file:")
display(metrics_df)

print("\nWeights by file:")
display(weights_df_all)

print("\nPowers used by each feature in each model:")
display(powers_by_model_df)

print("\nFeature importance by file:")
display(feature_importance_all_df)

if not skipped_df.empty:
    print("\nSkipped files:")
    display(skipped_df)

Model performance by file:


,file,rows_used,train_rows,test_rows,accuracy
0,cleaned_coin_Tether.csv,2317,1737,580,0.998276
1,cleaned_coin_USDCoin.csv,1001,750,251,0.956175
2,cleaned_coin_Bitcoin.csv,2990,2242,748,0.874332
3,cleaned_coin_Dogecoin.csv,2759,2069,690,0.865217
4,cleaned_coin_XRP.csv,2892,2169,723,0.854772
5,cleaned_coin_Monero.csv,2601,1950,651,0.844854
6,cleaned_coin_Stellar.csv,2526,1894,632,0.829114
7,cleaned_coin_Litecoin.csv,2990,2242,748,0.810160
8,cleaned_coin_NEM.csv,2287,1715,572,0.809441
9,cleaned_coin_EOS.csv,1465,1098,367,0.806540



Weights by file:


,file,intercept,w_open,w_high,w_low,w_close,w_volume
0,cleaned_coin_Aave.csv,-0.765668,-0.199454,-1.999531,-0.783236,2.383774,0.398043
1,cleaned_coin_BinanceCoin.csv,-1.611358,0.120392,1.624581,-2.081211,-0.331192,0.127914
2,cleaned_coin_Bitcoin.csv,-2.151859,0.145786,2.235541,-2.469585,0.357379,-0.215482
3,cleaned_coin_Cardano.csv,-1.480504,0.427482,0.139576,-1.378171,0.772505,0.076388
4,cleaned_coin_ChainLink.csv,-1.192941,0.024489,1.250874,-1.381310,-0.405230,0.283235
5,cleaned_coin_Cosmos.csv,-1.482095,1.108876,0.648592,-0.804908,-1.168255,0.069107
6,cleaned_coin_CryptocomCoin.csv,-1.900911,-0.371497,1.337541,-0.916961,-0.067682,-0.372132
7,cleaned_coin_Dogecoin.csv,-1.753045,-0.768094,4.318234,-2.905493,-0.410691,-0.201507
8,cleaned_coin_EOS.csv,-1.572841,0.480348,1.679163,-1.817003,-0.233025,-0.265426
9,cleaned_coin_Ethereum.csv,-1.510460,-1.305192,5.036682,-2.243282,-1.503367,-0.182094



Powers used by each feature in each model:


,file,feature,power_used,weight
0,cleaned_coin_Aave.csv,intercept,0,-0.765668
1,cleaned_coin_Aave.csv,open,1,-0.199454
2,cleaned_coin_Aave.csv,high,1,-1.999531
3,cleaned_coin_Aave.csv,low,1,-0.783236
4,cleaned_coin_Aave.csv,close,1,2.383774
...,...,...,...,...
133,cleaned_coin_XRP.csv,open,1,0.481185
134,cleaned_coin_XRP.csv,high,1,0.425137
135,cleaned_coin_XRP.csv,low,1,-1.191568
136,cleaned_coin_XRP.csv,close,1,0.375688



Feature importance by file:


,file,rank,feature,weight,abs_weight,importance_score_percent
0,cleaned_coin_Aave.csv,1,close,2.383774,2.383774,41.355973
1,cleaned_coin_Aave.csv,2,high,-1.999531,1.999531,34.689760
2,cleaned_coin_Aave.csv,3,low,-0.783236,0.783236,13.588313
3,cleaned_coin_Aave.csv,4,volume,0.398043,0.398043,6.905635
4,cleaned_coin_Aave.csv,5,open,-0.199454,0.199454,3.460319
...,...,...,...,...,...,...
110,cleaned_coin_XRP.csv,1,low,-1.191568,1.191568,48.010514
111,cleaned_coin_XRP.csv,2,open,0.481185,0.481185,19.387856
112,cleaned_coin_XRP.csv,3,high,0.425137,0.425137,17.129565
113,cleaned_coin_XRP.csv,4,close,0.375688,0.375688,15.137171
